# Cover & metadata augmentation for the Streamlit app

This notebook enriches `items.csv` so the Streamlit app can display book covers.

## Strategy

The app only displays books in users' recommendations or reading history. We
enrich **only that visible subset** to keep the pipeline practical.

### Cover lookup pipeline (Open Library only - free, no API key)

1. **Open Library Covers by ISBN** - HEAD request, strict validation
   (`?default=false` so missing covers return 404, not a 1x1 placeholder).
2. **Open Library Search API by ISBN (extended)** - collects up to 3
   `cover_i` values, plus LCCN and OCLC identifiers from the matched edition.
   Tries each as a different cover URL.
3. **Open Library Search API by Title+Author** - last-resort fallback for
   books still missing (including books with no ISBN at all).

All steps run in parallel with `ThreadPoolExecutor`. No API key, no quota.
Total runtime: ~15-25 minutes for ~5-8k visible books.

In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## 1. Load data

In [ ]:
BASE = "https://raw.githubusercontent.com/Trickwillfrit/EPFL_ROLEX_DATA/main"
ITEMS_URL = f"{BASE}/items.csv"
SUBMISSION_URL = f"{BASE}/submission.csv"
INTERACTIONS_URL = f"{BASE}/interactions_train.csv"

items = pd.read_csv(ITEMS_URL)
submission = pd.read_csv(SUBMISSION_URL)
interactions = pd.read_csv(INTERACTIONS_URL)

print("items shape:        ", items.shape)
print("submission shape:   ", submission.shape)
print("interactions shape: ", interactions.shape)
items.head(3)

## 2. Clean ISBNs - keep ALL of them per book

Each `ISBN Valid` field can hold several ISBNs separated by `;`. Keeping all
of them lets us try each form (10-digit, 13-digit, edition variants).

In [ ]:
def clean_isbn_list(raw):
    """Split a raw ISBN string into a clean list of ISBNs (length 10 or 13)."""
    if pd.isna(raw):
        return []
    parts = str(raw).replace(",", ";").split(";")
    cleaned = []
    for p in parts:
        c = "".join(ch for ch in p if ch.isdigit() or ch in "Xx")
        if len(c) in (10, 13):
            cleaned.append(c)
    seen, out = set(), []
    for c in cleaned:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

items["isbns_clean"] = items["ISBN Valid"].apply(clean_isbn_list)
items["isbn_clean"] = items["isbns_clean"].apply(lambda xs: xs[0] if xs else pd.NA)

n_with_isbn = items["isbns_clean"].apply(len).gt(0).sum()
print(f"Books with at least one valid ISBN: {n_with_isbn} / {len(items)}")
print(f"Books with NO ISBN (will use title+author): {len(items) - n_with_isbn}")

## 3. Restrict to "visible" books

The app only shows books that appear in some user's top-10 recommendations or
in their reading history. Everything else is invisible and not worth a network
call.

In [ ]:
rec_ids = set()
for s in submission["recommendation"].dropna():
    rec_ids.update(int(x) for x in str(s).split())

sub_users = set(submission["user_id"])
hist_ids = set(interactions.loc[interactions["u"].isin(sub_users), "i"])

visible_ids = rec_ids | hist_ids

items["is_visible"] = items["i"].isin(visible_ids)

print(f"Recommended items:           {len(rec_ids)}")
print(f"History items (sub users):   {len(hist_ids)}")
print(f"Visible items (union):       {len(visible_ids)} / {len(items)} total")
print(f"Books we will enrich:        {items['is_visible'].sum()}")

## 4. HTTP session and image validator

In [ ]:
def make_session():
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.3,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["HEAD", "GET"])
    adapter = HTTPAdapter(max_retries=retry, pool_connections=64, pool_maxsize=64)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": "EPFL-Library-Recsys/1.0 (student project)"})
    return s

session = make_session()

def is_real_image(url, timeout=5):
    """HEAD-validate that a URL returns a real image, not a 1x1 placeholder."""
    try:
        r = session.head(url, timeout=timeout, allow_redirects=True)
        return (r.status_code == 200 and
                r.headers.get("Content-Type", "").startswith("image/"))
    except requests.RequestException:
        return False

## 5. Source 1 - Open Library Covers by ISBN

`?default=false` makes Open Library return HTTP 404 when no real cover exists,
so we never accidentally record a 1x1 transparent placeholder URL.

In [ ]:
def openlibrary_isbn_url(isbn):
    return f"https://covers.openlibrary.org/b/isbn/{isbn}-M.jpg?default=false"

def check_openlibrary_isbn(isbn):
    url = openlibrary_isbn_url(isbn)
    return (isbn, url) if is_real_image(url) else (isbn, None)

visible_isbns = set()
for isbns in items.loc[items["is_visible"], "isbns_clean"]:
    visible_isbns.update(isbns)
visible_isbns = list(visible_isbns)
print(f"ISBNs to validate: {len(visible_isbns)}")

ol_isbn_map = {}
with ThreadPoolExecutor(max_workers=24) as pool:
    futures = [pool.submit(check_openlibrary_isbn, isbn) for isbn in visible_isbns]
    for i, fut in enumerate(as_completed(futures), 1):
        isbn, url = fut.result()
        ol_isbn_map[isbn] = url
        if i % 1000 == 0:
            n = sum(1 for v in ol_isbn_map.values() if v)
            print(f"  {i}/{len(visible_isbns)} checked, {n} hits")

n_hits = sum(1 for v in ol_isbn_map.values() if v)
print(f"\nOpen Library Covers (ISBN): {n_hits} / {len(visible_isbns)}")

In [ ]:
def first_ol_isbn_cover(isbns):
    for isbn in isbns:
        url = ol_isbn_map.get(isbn)
        if url:
            return url
    return pd.NA

items["cover_ol_isbn"] = items["isbns_clean"].apply(first_ol_isbn_cover)
n_v = items["is_visible"].sum()
n_ok = (items["is_visible"] & items["cover_ol_isbn"].notna()).sum()
print(f"Visible books covered: {n_ok} / {n_v} ({100*n_ok/n_v:.1f}%)")

## 6. Source 2 - Open Library Search API (extended)

For each remaining ISBN, query the Search API and collect:
- up to 3 candidate `cover_i` values (different editions),
- all returned LCCN identifiers,
- all returned OCLC identifiers.

Each is built into a different cover URL. We try them all and keep the first
that returns a real image. LCCN/OCLC lookups frequently succeed for older
European books where the ISBN-based covers endpoint is empty.

In [ ]:
def openlibrary_search_by_isbn(isbn):
    """Return (isbn, url_or_None) using extended Open Library Search."""
    try:
        r = session.get(
            "https://openlibrary.org/search.json",
            params={"isbn": isbn, "limit": 3,
                    "fields": "cover_i,lccn,oclc,edition_key"},
            timeout=8,
        )
        if r.status_code != 200:
            return isbn, None
        docs = r.json().get("docs") or []
        if not docs:
            return isbn, None

        # Build candidate URLs from cover_i / lccn / oclc across all docs
        candidates = []
        for d in docs:
            ci = d.get("cover_i")
            if ci:
                candidates.append(f"https://covers.openlibrary.org/b/id/{ci}-M.jpg?default=false")
            for lccn in (d.get("lccn") or [])[:2]:
                candidates.append(f"https://covers.openlibrary.org/b/lccn/{lccn}-M.jpg?default=false")
            for oclc in (d.get("oclc") or [])[:2]:
                candidates.append(f"https://covers.openlibrary.org/b/oclc/{oclc}-M.jpg?default=false")

        for url in candidates:
            if is_real_image(url):
                return isbn, url
        return isbn, None
    except requests.RequestException:
        return isbn, None

In [ ]:
missing_mask_1 = items["is_visible"] & items["cover_ol_isbn"].isna()
missing_isbns_1 = set()
for isbns in items.loc[missing_mask_1, "isbns_clean"]:
    missing_isbns_1.update(isbns)
missing_isbns_1 = list(missing_isbns_1)
print(f"Visible books still missing: {missing_mask_1.sum()}")
print(f"ISBNs to query against Search API: {len(missing_isbns_1)}")

ol_search_map = {}
with ThreadPoolExecutor(max_workers=12) as pool:
    futures = [pool.submit(openlibrary_search_by_isbn, isbn) for isbn in missing_isbns_1]
    for i, fut in enumerate(as_completed(futures), 1):
        isbn, url = fut.result()
        ol_search_map[isbn] = url
        if i % 500 == 0:
            n = sum(1 for v in ol_search_map.values() if v)
            print(f"  {i}/{len(missing_isbns_1)} checked, {n} hits")

n = sum(1 for v in ol_search_map.values() if v)
print(f"\nSearch API (extended): {n} / {len(missing_isbns_1)} ISBNs found")

In [ ]:
def first_ol_search_cover(isbns):
    for isbn in isbns:
        url = ol_search_map.get(isbn)
        if url:
            return url
    return pd.NA

items["cover_ol_search"] = items["isbns_clean"].apply(first_ol_search_cover)
n_ok = (items["is_visible"] & items["cover_ol_search"].notna()).sum()
print(f"Visible books with cover via Search API: {n_ok}")

## 7. Source 3 - Open Library Title+Author search (last-resort)

For books still missing - including books with no ISBN at all - search by
title and author. False positives exist but title+author is reasonably
precise. We strictly validate the resulting URL with `is_real_image`.

In [ ]:
def clean_query(s, max_len=60):
    """Strip noisy punctuation from titles/authors before querying."""
    if pd.isna(s):
        return ""
    s = str(s).split(":")[0].split("/")[0]  # drop subtitles after : or /
    keep = " -\'\u2019\u00e9\u00e8\u00ea\u00e0\u00e2\u00e7\u00f4\u00ee\u00fb\u00ef\u00fc"
    s = "".join(ch if ch.isalnum() or ch in keep else " " for ch in s)
    return " ".join(s.split())[:max_len]

def openlibrary_title_author(item_id, title, author):
    """Return (item_id, url_or_None) using Search API by title+author."""
    title_q = clean_query(title)
    author_q = clean_query(author, max_len=40)
    if not title_q:
        return item_id, None
    try:
        params = {"title": title_q, "limit": 1, "fields": "cover_i,lccn,oclc"}
        if author_q:
            params["author"] = author_q
        r = session.get("https://openlibrary.org/search.json",
                        params=params, timeout=8)
        if r.status_code != 200:
            return item_id, None
        docs = r.json().get("docs") or []
        if not docs:
            return item_id, None
        d = docs[0]
        candidates = []
        if d.get("cover_i"):
            candidates.append(
                f"https://covers.openlibrary.org/b/id/{d['cover_i']}-M.jpg?default=false")
        for lccn in (d.get("lccn") or [])[:1]:
            candidates.append(
                f"https://covers.openlibrary.org/b/lccn/{lccn}-M.jpg?default=false")
        for oclc in (d.get("oclc") or [])[:1]:
            candidates.append(
                f"https://covers.openlibrary.org/b/oclc/{oclc}-M.jpg?default=false")
        for url in candidates:
            if is_real_image(url):
                return item_id, url
        return item_id, None
    except requests.RequestException:
        return item_id, None

In [ ]:
missing_mask_2 = (
    items["is_visible"]
    & items["cover_ol_isbn"].isna()
    & items["cover_ol_search"].isna()
)
missing_books = items.loc[missing_mask_2, ["i", "Title", "Author"]].copy()
print(f"Visible books still missing after Search API: {len(missing_books)}")

ol_title_map = {}  # item_id -> url or None
with ThreadPoolExecutor(max_workers=12) as pool:
    futures = [
        pool.submit(openlibrary_title_author, row["i"], row["Title"], row["Author"])
        for _, row in missing_books.iterrows()
    ]
    for i, fut in enumerate(as_completed(futures), 1):
        item_id, url = fut.result()
        ol_title_map[item_id] = url
        if i % 500 == 0:
            n = sum(1 for v in ol_title_map.values() if v)
            print(f"  {i}/{len(missing_books)} checked, {n} hits")

n = sum(1 for v in ol_title_map.values() if v)
print(f"\nTitle+Author search: {n} / {len(missing_books)} books found")

items["cover_ol_title"] = items["i"].map(ol_title_map)

## 8. Combine sources and report final coverage

In [ ]:
items["cover_url"] = (
    items["cover_ol_isbn"]
    .combine_first(items["cover_ol_search"])
    .combine_first(items["cover_ol_title"])
)

n_total = len(items)
n_visible = items["is_visible"].sum()
n_ol1 = (items["is_visible"] & items["cover_ol_isbn"].notna()).sum()
n_ol2 = (items["is_visible"] & items["cover_ol_isbn"].isna()
         & items["cover_ol_search"].notna()).sum()
n_ol3 = (items["is_visible"] & items["cover_ol_isbn"].isna()
         & items["cover_ol_search"].isna()
         & items["cover_ol_title"].notna()).sum()
n_visible_ok = (items["is_visible"] & items["cover_url"].notna()).sum()

print("=== Coverage report (visible books only) ===")
print(f"Visible books:                    {n_visible}")
print(f"  via Open Library Covers (ISBN): {n_ol1}")
print(f"  via Open Library Search API:    {n_ol2}")
print(f"  via Title+Author search:        {n_ol3}")
print(f"  WITH cover (any source):        {n_visible_ok} / {n_visible} "
      f"({100*n_visible_ok/n_visible:.1f}%)")
print(f"  WITHOUT cover:                  {n_visible - n_visible_ok}")
print()
print(f"Catalog-wide books with cover:    {items['cover_url'].notna().sum()} / {n_total}")
print("(Catalog-wide is lower because we only enriched visible books - intended.)")

## 9. Save `items_app.csv`

In [ ]:
items_app = items[[
    "i",
    "Title",
    "Author",
    "Publisher",
    "Subjects",
    "ISBN Valid",
    "isbn_clean",
    "cover_url",
]].copy()

print("items_app shape:", items_app.shape)
items_app.head()

In [ ]:
items_app.to_csv("items_app.csv", index=False)
print("Saved items_app.csv")

# In Colab, also download to your machine:
# from google.colab import files
# files.download("items_app.csv")

## 10. (Optional) Spot-check a few covers

In [ ]:
from IPython.display import Image, display

n_with_cover = items_app["cover_url"].notna().sum()
sample = items_app[items_app["cover_url"].notna()].sample(
    min(8, n_with_cover), random_state=42
)
for _, row in sample.iterrows():
    print(f"[{row['i']}] {row['Title'][:60]}")
    try:
        display(Image(url=row["cover_url"], width=120))
    except Exception as e:
        print("  (could not render)", e)